In [ ]:
import json
import pickle
import numpy as np
import os
import matplotlib.pyplot as plt


def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def calculate_eta(vi, wi, theta, V):
    rho_v = theta[vi, :]          # ρ_v = θ_v
    alpha_w = theta[wi + V, :]    # α_w = θ_{V + w}
    eta = rho_v.dot(alpha_w)      # η = ρ_v^T α_w
    return eta

def extract_correlation(word1, word2, vocabulary, theta_samples):
    vi = vocabulary[word1]
    wi = vocabulary[word2]
    V = len(vocabulary)

    p_samples = []
    for theta in theta_samples:
        eta = calculate_eta(vi, wi, theta, V)
        p_samples.append(sigmoid(eta))

    return np.array(p_samples)


def credible_interval(samples, confidence=0.90):
    lower_bound = np.percentile(samples, (1-confidence)/ 2 * 100)
    upper_bound = np.percentile(samples, (1+confidence)/ 2 * 100)
    return lower_bound, upper_bound

def load_fit(filepath):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

In [ ]:
data_path = '100k_v10_d2_.json'
#model_path = os.path.join('models', 'word2vec_fix.stan')
save_dir = 'stan_fits'

with open('100k_v10_d2_.json') as f:
    data = json.load(f)

true_theta = np.array(data['theta'])
vocabulary = data['vocabulary']


# Word - Word similarity plot

In [ ]:


def calculate_true_correlation(word1, word2, vocabulary, true_theta):
    vi = vocabulary[word1]
    wi = vocabulary[word2]
    V = len(vocabulary)
    eta = calculate_eta(vi, wi, true_theta, V)
    return eta

def extract_correlation(word1, word2, vocabulary, theta_samples):
    """
    Calculate the correlation between the two specified words over all samples.
    """
    vi = vocabulary[word1]
    wi = vocabulary[word2]
    V = len(vocabulary)

    eta_samples = []
    for theta in theta_samples:
        eta = calculate_eta(vi, wi, theta, V)
        eta_samples.append(eta)

    return np.array(eta_samples)

def credible_interval(samples, confidence=0.90):
    lower_bound = np.percentile(samples, (1 - confidence) / 2 * 100)
    upper_bound = np.percentile(samples, (1 + confidence) / 2 * 100)
    return lower_bound, upper_bound


with open('100k_v10_d2_.json') as f:
    data = json.load(f)

true_theta = np.array(data['theta'])
vocabulary = data['vocabulary']


# Directory where fits are saved
#output_dir = "stan_fit_nofix"
output_dir = "stan_fit_nofix"

# Sizes to iterate over
sizes = [100, 200, 500, 1000, 5000, 10000, 20000, 50000, 100000]

# Words to compare
word1 = "word5"
word2 = "word6"

# true correlation
true_correlation = calculate_true_correlation(word1, word2, vocabulary, true_theta)


correlations = []
credible_intervals = []

for size in sizes:
    filename = os.path.join(output_dir, f'stan_fit_{size}.pkl')

    with open(filename, 'rb') as f:
        fit = pickle.load(f)

    # posterior samples
    estimated_word_vectors = fit['word_vectors']
    estimated_context_vectors = fit['context_vectors']
    num_samples = estimated_word_vectors.shape[2]

    theta_samples = []
    for i in range(num_samples):
        word_vectors = estimated_word_vectors[:, :, i]
        context_vectors = estimated_context_vectors[:, :, i]
        theta = np.vstack((word_vectors, context_vectors))
        theta_samples.append(theta)
    
    eta_samples = extract_correlation(word1, word2, vocabulary, theta_samples)

    mean_corr = np.mean(eta_samples)
    lower_ci, upper_ci = credible_interval(eta_samples)

    correlations.append(mean_corr)
    credible_intervals.append((lower_ci, upper_ci))

    print(f"Size {size}: Mean correlation = {mean_corr}, 90% CI = ({lower_ci}, {upper_ci})")

correlations = np.array(correlations)
credible_intervals = np.array(credible_intervals)

plt.figure(figsize=(10, 6))
plt.ylim([0.4, 0.8])
plt.plot(sizes, correlations, marker='o', linestyle='-', label='HMC', color='tab:brown')
plt.fill_between(sizes, credible_intervals[:, 0], credible_intervals[:, 1], color='tab:brown', alpha=0.2, label='HMC 90% CI')
plt.axhline(y=true_correlation, color='r', linestyle='--', label='True')
plt.xlabel('Data Size')
plt.ylabel('Co-occurence Probability')
#plt.title('Stan, %s - %s'%(word1, word2))
plt.legend()
plt.grid(True)
plt.show()


# Posterior sample plot for individual word (donut plot)

In [ ]:
import seaborn as sns
import pandas as pd


with open('100k_v10_d2_.json') as f:
    data = json.load(f)

vocabulary = data['vocabulary']

# specify fit.
fit_filepath = os.path.join('stan_fit_dxdfix', 'stan_fit_fixed_100000.pkl')
fit_filepath = os.path.join('stan_fit_nofix', 'stan_fit_50000.pkl')
fit = load_fit(fit_filepath)

word_vectors_samples = fit['word_vectors']
context_vectors_samples = fit['context_vectors']


def plot_posterior(word_index, word_vectors_samples, context_vectors_samples, vocabulary):
    # Extract samples for the given word
    word_samples = word_vectors_samples[word_index]
    #context_samples = context_vectors_samples[context_index]

    # for seaborn..
    data = {
        'Dim 1': word_samples[0, :],
        'Dim 2': word_samples[1, :],
        #'Context Dim 1': context_samples[0, :],
        #'Context Dim 2': context_samples[1, :],
    }
    df = pd.DataFrame(data)


    plt.figure(figsize=(10, 6))
    sns.jointplot(x='Dim 1', y='Dim 2', data=df, kind='kde', fill=True)
    
    plt.suptitle(f"Posterior samples for  '{list(vocabulary.keys())[word_index]}'", y=1.02)
    plt.scatter([np.mean(word_samples[0, :])] , [np.mean(word_samples[1, :])] , marker ='x', color='tab:red')
    plt.axhline(y=0, color='tab:gray', ls='--')
    plt.axvline(x=0, color='tab:gray', ls='--')
    plt.show()

word_index = 4 # index of the word to plot


plot_posterior(word_index, word_vectors_samples, context_vectors_samples, vocabulary)
